In [ ]:
# Cell 1: Check GPU
!nvidia-smi

# Cell 2: Install dependencies
!pip install ultralytics roboflow -q

# Cell 3: Imports
from ultralytics import YOLO
import torch
import os
import shutil
from google.colab import drive, files
import cv2
from PIL import Image
import matplotlib.pyplot as plt
print(f"GPU: {torch.cuda.is_available()}")

# Cell 4: Mount Drive (folder already exists)
drive.mount('/content/drive')
save_path = '/content/drive/MyDrive/PPE_Detection'
print(f"Save path: {save_path}")

Mon Apr 20 04:34:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 5: Get dataset from Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key="API-KEY")
project = rf.workspace("rohit-yadav-fswnv").project("safety-tucje-ba78y")
dataset = project.version(1).download("yolov8")

dataset_path = dataset.location
print(f"Dataset: {dataset_path}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Safety-1 in yolov8:: 100%|██████████| 52201/52201 [00:16<00:00, 3130.09it/s] 


Dataset: /content/Safety-1


In [ ]:
# Cell 6: Load model
model = YOLO('yolov8n.pt')

# Cell 7: Class weights (no-vest and vest get higher weight)
# person(7275), helmet(7714), no-helmet(7855), vest(5559), no-vest(4785)
class_weights = [0.91, 0.86, 0.85, 1.20, 1.39]  # person, helmet, no-helmet, vest, no-vest
print(f"Weights: {class_weights}")

# Cell 8: Train
results = model.train(
    data=f'{dataset_path}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project=f'{save_path}/runs',
    name='weighted_training',
    exist_ok=True
)

Weights: [0.91, 0.86, 0.85, 1.2, 1.39]
Ultralytics 8.4.39 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Safety-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=weighted_training, nbs=64, nms=False, opset=None, optimize=False, op